# White-Box Lie Detection - Multi-Model Experiments (Colab)

Ce notebook lance les experiences de lie detection white-box sur des **gros modeles LLM** en utilisant le GPU de Google Colab.

**Projet base sur** [llm_lie_detection_black_vs_white_box](https://github.com/GeorgeVasile04/llm_lie_detection_black_vs_white_box) par GeorgeVasile04.

## Comment utiliser
1. Ouvre ce notebook dans Google Colab
2. Va dans **Runtime > Change runtime type > GPU (T4)** (ou **A100** si Colab Pro)
3. **IMPORTANT** : Accepte les licences sur HuggingFace avant de lancer :
   - Gemma : https://huggingface.co/google/gemma-3-27b-it → "Accept"
   - Llama (si Pro) : https://huggingface.co/meta-llama/Llama-2-70b-chat-hf → "Accept"
   - Cree un token sur https://huggingface.co/settings/tokens
4. Execute toutes les cellules dans l'ordre

## Modeles testes

### Tier Gratuit (T4 - 15GB VRAM)
| Modele | Params | VRAM (4-bit) |
|--------|--------|-------------|
| Gemma 3 27B | 27B | ~15GB |
| Gemma 3 12B | 12B | ~8GB |
| Gemma 3 4B | 4B | ~3GB |

### Tier Pro (A100 - 40GB VRAM) - 10 euros/mois
| Modele | Params | VRAM (4-bit) |
|--------|--------|-------------|
| Llama-2-70b-chat | 70B | ~38GB |

## 1. Setup: GPU check + installations

In [ ]:
# Verifier le GPU disponible
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    gpu_vram = props.total_memory / 1024**3
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {gpu_vram:.1f} GB")
    if gpu_vram < 14:
        print("ATTENTION: Moins de 14GB VRAM - le modele 27B risque de ne pas tenir.")
        print("Tu peux le commenter dans la config et garder seulement 12B + 4B.")
else:
    print("PAS DE GPU! Va dans Runtime > Change runtime type > GPU")
    raise RuntimeError("GPU requis")

In [ ]:
# Installer les dependances (transformers >= 4.49 requis pour Gemma 3)
!pip install -U -q "transformers>=4.49" accelerate bitsandbytes datasets scikit-learn matplotlib pandas numpy tqdm huggingface_hub

# IMPORTANT: Redemarrer le runtime apres l'install pour que numpy se recharge correctement
# Ca va couper l'execution ici - c'est NORMAL, relance ensuite a partir de la cellule suivante
import os
if not os.environ.get("DEPS_INSTALLED"):
    os.environ["DEPS_INSTALLED"] = "1"
    print("=" * 50)
    print("  Dependances installees!")
    print("  Le runtime va redemarrer automatiquement.")
    print("  -> Relance TOUTES les cellules apres le restart")
    print("  (Runtime > Run all, ou Ctrl+F9)")
    print("=" * 50)
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)
else:
    print("Runtime deja redemarre, on continue...")

import transformers
version = transformers.__version__
major, minor = int(version.split(".")[0]), int(version.split(".")[1])
print(f"transformers version: {version}")
if major < 4 or (major == 4 and minor < 49):
    raise RuntimeError("transformers >= 4.49 requis pour Gemma 3! Relance le runtime.")
print("OK - version compatible avec Gemma 3")

In [ ]:
# Monter Google Drive pour sauvegarder les resultats
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_DIR = '/content/drive/MyDrive/LLM_Lie_Detection_Results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Resultats sauvegardes dans: {RESULTS_DIR}")

In [ ]:
# Login HuggingFace - OBLIGATOIRE pour les modeles Gemma (licence Google)
# 1. Cree un token sur https://huggingface.co/settings/tokens (type "Read")
# 2. Accepte la licence sur https://huggingface.co/google/gemma-3-27b-it
# 3. Colle ton token ci-dessous quand demande
from huggingface_hub import login
login()

## 2. Code principal: extraction d'activations + probes

In [ ]:
import gc
import json
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset

print("Imports OK")

In [ ]:
#######################################################################
# CONFIGURATION
#######################################################################

# Modeles du plus gros au plus petit
MODELS_TO_TEST = [
    # --- Tier Gratuit (T4, 15GB VRAM) ---
    {"name": "gemma-3-27b-it", "hf_id": "google/gemma-3-27b-it", "num_params": 27.0, "quant": "4bit"},
    {"name": "gemma-3-12b-it", "hf_id": "google/gemma-3-12b-it", "num_params": 12.0, "quant": "4bit"},
    {"name": "gemma-3-4b-it",  "hf_id": "google/gemma-3-4b-it",  "num_params": 4.0,  "quant": "4bit"},

    # --- Decommenter si Colab Pro (A100 40GB) ---
    # {"name": "Llama-2-70b-chat", "hf_id": "meta-llama/Llama-2-70b-chat-hf", "num_params": 70.0, "quant": "4bit"},
]

# Datasets (les memes que l'experience locale)
DATASETS_CONFIG = [
    {"name": "arc_easy",  "hf_id": "allenai/ai2_arc", "subset": "ARC-Easy", "type": "arc"},
    {"name": "boolq",     "hf_id": "google/boolq",    "subset": None,       "type": "boolq"},
    {"name": "imdb",      "hf_id": "stanfordnlp/imdb", "subset": None,      "type": "imdb"},
]

MAX_TRAIN = 100
MAX_VAL = 200
LAYERS_SKIP = 2  # Extraire 1 couche sur 2 pour aller plus vite

print(f"Modeles a tester: {[m['name'] for m in MODELS_TO_TEST]}")
print(f"Datasets: {[d['name'] for d in DATASETS_CONFIG]}")
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU VRAM: {vram_gb:.1f} GB")

In [ ]:
#######################################################################
# Dataset Loading - cree des paires Vrai/Faux
#######################################################################

def load_binary_dataset(config):
    """Load a dataset and convert to True/False text statements."""
    ds_type = config["type"]
    samples = {"train": [], "validation": []}

    if ds_type == "arc":
        ds = load_dataset(config["hf_id"], config["subset"])
        for split_name, split_key in [("train", "train"), ("validation", "test")]:
            split = ds[split_key]
            for row in split:
                q = row["question"]
                choices = row["choices"]["text"]
                labels = row["choices"]["label"]
                correct_idx = labels.index(row["answerKey"]) if row["answerKey"] in labels else 0
                correct = choices[correct_idx]
                wrong = choices[(correct_idx + 1) % len(choices)]
                group_id = q[:50]
                samples[split_name].append({"text": f"Question: {q}\nAnswer: {correct}\nIs this answer correct? Yes.", "label": True, "group_id": group_id})
                samples[split_name].append({"text": f"Question: {q}\nAnswer: {wrong}\nIs this answer correct? No.", "label": False, "group_id": group_id})

    elif ds_type == "boolq":
        ds = load_dataset(config["hf_id"])
        for split_name in ["train", "validation"]:
            split = ds[split_name]
            for row in split:
                answer = "Yes" if row["answer"] else "No"
                samples[split_name].append({"text": f"Passage: {row['passage'][:300]}\nQuestion: {row['question']}\nAnswer: {answer}", "label": row["answer"], "group_id": None})

    elif ds_type == "imdb":
        ds = load_dataset(config["hf_id"])
        for split_name, split_key in [("train", "train"), ("validation", "test")]:
            split = ds[split_key]
            for row in split:
                sentiment = "positive" if row["label"] == 1 else "negative"
                samples[split_name].append({"text": f"Review: {row['text'][:400]}\nSentiment: {sentiment}", "label": row["label"] == 1, "group_id": None})

    # Limit and shuffle
    for split in ["train", "validation"]:
        np.random.seed(42)
        np.random.shuffle(samples[split])
        limit = MAX_TRAIN if split == "train" else MAX_VAL
        samples[split] = samples[split][:limit]

    return samples

In [ ]:
#######################################################################
# Model Loading with 4-bit quantization
#######################################################################

def load_model(model_config):
    """Load a model with 4-bit quantization for GPU inference."""
    hf_id = model_config["hf_id"]
    print(f"Loading {hf_id}...")

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

    model = AutoModelForCausalLM.from_pretrained(
        hf_id,
        quantization_config=quant_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(hf_id, trust_remote_code=True)

    # Fix: certains tokenizers n'ont pas de pad_token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Trouver les couches transformer automatiquement
    layers = None
    for attr_path in ["model.layers", "transformer.h", "gpt_neox.layers"]:
        obj = model
        try:
            for part in attr_path.split("."):
                obj = getattr(obj, part)
            layers = obj
            print(f"  Found {len(layers)} layers via {attr_path}")
            break
        except AttributeError:
            continue

    if layers is None:
        raise ValueError(f"Cannot find transformer layers in {hf_id}")

    model.eval()
    vram_used = torch.cuda.memory_allocated() / 1024**3
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"  VRAM used: {vram_used:.1f} GB / {vram_total:.1f} GB")

    return model, tokenizer, layers

In [ ]:
#######################################################################
# Activation Extraction
#######################################################################

@torch.inference_mode()
def extract_activations(model, tokenizer, layers, text, layers_skip=2):
    """Extract hidden-state activations at each layer for the last token."""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    # Envoyer sur GPU (cuda) directement - plus fiable que model.device avec device_map
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

    layer_indices = list(range(0, len(layers), layers_skip))
    activations = {}
    hooks = []

    for idx in layer_indices:
        name = f"h{idx}"
        def make_hook(layer_name):
            def hook_fn(module, input, output):
                if isinstance(output, tuple):
                    hidden = output[0]
                else:
                    hidden = output
                activations[layer_name] = hidden[0, -1, :].detach().float().cpu().numpy()
            return hook_fn

        h = layers[idx].register_forward_hook(make_hook(name))
        hooks.append(h)

    model(**inputs)

    for h in hooks:
        h.remove()

    return activations

In [ ]:
#######################################################################
# Probe Algorithms (same 6 as the original project)
#######################################################################

@dataclass
class ProbeResult:
    direction: np.ndarray  # The learned "truth direction"

    def predict(self, activations):
        logits = activations @ self.direction
        return logits


def train_dim(activations, labels):
    """Difference in Means - the simplest and often best probe."""
    true_mean = activations[labels].mean(axis=0)
    false_mean = activations[~labels].mean(axis=0)
    direction = true_mean - false_mean
    direction = direction / (np.linalg.norm(direction) + 1e-8)
    return ProbeResult(direction=direction)


def train_lda(activations, labels):
    """Linear Discriminant Analysis."""
    try:
        lda = LinearDiscriminantAnalysis()
        lda.fit(activations, labels)
        direction = lda.coef_[0]
        direction = direction / (np.linalg.norm(direction) + 1e-8)
        return ProbeResult(direction=direction)
    except Exception:
        return train_dim(activations, labels)


def train_lr(activations, labels):
    """Logistic Regression."""
    lr = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs')
    lr.fit(activations, labels)
    direction = lr.coef_[0]
    direction = direction / (np.linalg.norm(direction) + 1e-8)
    return ProbeResult(direction=direction)


def train_pca(activations, labels):
    """PCA - unsupervised, finds direction of max variance."""
    pca = PCA(n_components=1)
    pca.fit(activations)
    direction = pca.components_[0]
    direction = direction / (np.linalg.norm(direction) + 1e-8)
    return ProbeResult(direction=direction)


def train_lat(activations, labels):
    """Linear Artificial Tomography - PCA on random pair differences."""
    np.random.seed(42)
    n = len(activations)
    indices = np.random.permutation(n)
    half = n // 2
    diffs = activations[indices[:half]] - activations[indices[half:2*half]]
    pca = PCA(n_components=1)
    pca.fit(diffs)
    direction = pca.components_[0]
    direction = direction / (np.linalg.norm(direction) + 1e-8)
    return ProbeResult(direction=direction)


PROBE_ALGORITHMS = {
    "DIM": train_dim,
    "LDA": train_lda,
    "LR": train_lr,
    "PCA": train_pca,
    "LAT": train_lat,
}

print(f"Probe algorithms: {list(PROBE_ALGORITHMS.keys())}")

In [ ]:
#######################################################################
# Evaluation
#######################################################################

def evaluate_probe(probe, activations, labels):
    """Evaluate a probe, trying both directions (flipping if needed)."""
    logits = probe.predict(activations)
    preds = logits > 0
    acc = accuracy_score(labels, preds)
    acc_flipped = accuracy_score(labels, ~preds)
    return max(acc, acc_flipped)

## 3. Lancer les experiences

In [ ]:
#######################################################################
# MAIN EXPERIMENT LOOP
#######################################################################

all_results = []

for model_config in MODELS_TO_TEST:
    model_name = model_config["name"]
    num_params = model_config["num_params"]
    print(f"\n{'='*60}")
    print(f"  MODEL: {model_name} ({num_params}B params)")
    print(f"  Started: {time.strftime('%H:%M:%S')}")
    print(f"{'='*60}")

    model_dir = Path(RESULTS_DIR) / model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    # Load model
    start_time = time.time()
    try:
        model, tokenizer, layers = load_model(model_config)
    except Exception as e:
        print(f"  ERREUR chargement {model_name}: {e}")
        print(f"  On passe au modele suivant...")
        gc.collect()
        torch.cuda.empty_cache()
        continue

    num_layers = len(layers)

    # Extract activations for all datasets
    all_activations = {}
    for ds_config in DATASETS_CONFIG:
        ds_name = ds_config["name"]
        print(f"\n--- Dataset: {ds_name} ---")
        dataset = load_binary_dataset(ds_config)
        print(f"  Train: {len(dataset['train'])}, Val: {len(dataset['validation'])}")

        ds_activations = {"train": [], "validation": []}
        for split in ["train", "validation"]:
            print(f"  Extracting {split}...")
            errors = 0
            for sample in tqdm(dataset[split], desc=f"    {split}"):
                try:
                    acts = extract_activations(model, tokenizer, layers, sample["text"], LAYERS_SKIP)
                    ds_activations[split].append({
                        "activations": acts,
                        "label": sample["label"],
                        "group_id": sample.get("group_id"),
                    })
                except Exception as e:
                    errors += 1
                    if errors <= 2:
                        print(f"    Warning: {e}")
                    continue
            if errors > 0:
                print(f"    ({errors} erreurs ignorees)")

        all_activations[ds_name] = ds_activations
        print(f"  OK: {len(ds_activations['train'])} train, {len(ds_activations['validation'])} val")

    # Unload model to free VRAM
    del model, tokenizer, layers
    gc.collect()
    torch.cuda.empty_cache()
    vram_after = torch.cuda.memory_allocated() / 1024**3
    print(f"\nModel unloaded. VRAM: {vram_after:.1f} GB")

    # Train probes and evaluate
    print(f"\n--- Training probes ---")
    first_ds = list(all_activations.keys())[0]
    first_samples = all_activations[first_ds]["train"]
    if len(first_samples) == 0:
        print(f"  ERREUR: aucune activation extraite pour {model_name}, on passe...")
        continue
    layer_names = sorted(first_samples[0]["activations"].keys(), key=lambda x: int(x[1:]))

    for train_ds in all_activations:
        train_samples = all_activations[train_ds]["train"]
        for layer_name in layer_names:
            layer_idx = int(layer_name[1:])

            valid = [s for s in train_samples if layer_name in s["activations"]]
            if len(valid) < 10:
                continue
            X_train = np.stack([s["activations"][layer_name] for s in valid]).astype(np.float32)
            y_train = np.array([s["label"] for s in valid], dtype=bool)

            for algo_name, algo_fn in PROBE_ALGORITHMS.items():
                try:
                    probe = algo_fn(X_train, y_train)
                except Exception:
                    continue

                for eval_ds in all_activations:
                    val_samples = all_activations[eval_ds]["validation"]
                    valid_val = [s for s in val_samples if layer_name in s["activations"]]
                    if len(valid_val) < 10:
                        continue
                    X_val = np.stack([s["activations"][layer_name] for s in valid_val]).astype(np.float32)
                    y_val = np.array([s["label"] for s in valid_val], dtype=bool)

                    try:
                        acc = evaluate_probe(probe, X_val, y_val)
                    except Exception:
                        continue

                    all_results.append({
                        "model": model_name,
                        "num_params": num_params,
                        "train_dataset": train_ds,
                        "eval_dataset": eval_ds,
                        "algorithm": algo_name,
                        "layer": layer_idx,
                        "num_layers": num_layers,
                        "accuracy": acc,
                        "in_distribution": train_ds == eval_ds,
                    })

    elapsed = time.time() - start_time
    print(f"\n  {model_name} DONE in {elapsed/60:.1f} min ({len(all_results)} results total)")

    # Save intermediate results after each model
    df = pd.DataFrame(all_results)
    df.to_csv(f"{RESULTS_DIR}/results_all.csv", index=False)
    print(f"  Resultats sauvegardes dans {RESULTS_DIR}/results_all.csv")

print(f"\n\n{'='*60}")
print(f"  ALL MODELS DONE! {len(all_results)} results")
print(f"{'='*60}")

## 4. Visualisation des resultats

In [ ]:
# Charger les resultats
df = pd.read_csv(f"{RESULTS_DIR}/results_all.csv")
print(f"Total: {len(df)} results")
print(f"Modeles: {df['model'].unique()}")

# Tableau recapitulatif
summary = []
for model in df['model'].unique():
    m = df[df['model'] == model]
    m_in = m[m['in_distribution'] == True]
    m_cross = m[m['in_distribution'] == False]
    summary.append({
        'Model': model,
        'Layers': m['num_layers'].iloc[0],
        'In-Dist Acc': f"{m_in['accuracy'].mean():.1%}",
        'Cross-DS Acc': f"{m_cross['accuracy'].mean():.1%}",
        'Best Algo': m_in.groupby('algorithm')['accuracy'].mean().idxmax(),
        'Best Layer': m_in.groupby('layer')['accuracy'].mean().idxmax(),
    })

summary_df = pd.DataFrame(summary)
print("\n" + summary_df.to_string(index=False))
summary_df.to_csv(f"{RESULTS_DIR}/summary.csv", index=False)

In [ ]:
# Graphe 1: Accuracy par couche pour chaque modele
in_dist = df[df['in_distribution'] == True]
models = sorted(df['model'].unique())
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(models)))

fig, ax = plt.subplots(figsize=(12, 7))
for i, model_name in enumerate(models):
    model_data = in_dist[in_dist['model'] == model_name]
    layer_acc = model_data.groupby('layer')['accuracy'].mean()
    max_layer = layer_acc.index.max()
    ax.plot(layer_acc.index / max_layer, layer_acc.values,
            label=model_name, marker='o', markersize=4, linewidth=2, color=colors[i])

ax.set_xlabel('Relative Layer Position (0=first, 1=last)', fontsize=12)
ax.set_ylabel('Mean Accuracy', fontsize=12)
ax.set_title('Layer-wise Lie Detection Accuracy (Colab Models)', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_ylim(0.4, 1.0)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(f"{RESULTS_DIR}/layer_accuracy.png", dpi=150)
plt.show()

In [ ]:
# Graphe 2: Comparaison des algorithmes par modele
algorithms = sorted(df['algorithm'].unique())
n_models = len(models)
x = np.arange(len(algorithms))
width = 0.8 / max(n_models, 1)

fig, ax = plt.subplots(figsize=(14, 7))
for i, model_name in enumerate(models):
    model_data = in_dist[in_dist['model'] == model_name]
    algo_acc = model_data.groupby('algorithm')['accuracy'].mean()
    vals = [algo_acc.get(a, 0) for a in algorithms]
    ax.bar(x + i * width, vals, width, label=model_name, color=colors[i])

ax.set_xlabel('Probe Algorithm', fontsize=12)
ax.set_ylabel('Mean Accuracy', fontsize=12)
ax.set_title('Algorithm Performance by Model', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * (n_models - 1) / 2)
ax.set_xticklabels(algorithms)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_ylim(0.4, 1.0)
ax.grid(True, alpha=0.3, axis='y')
fig.tight_layout()
fig.savefig(f"{RESULTS_DIR}/algorithm_comparison.png", dpi=150)
plt.show()

In [ ]:
# Graphe 3: Matrice de generalisation pour chaque modele
for model_name in models:
    model_data = df[df['model'] == model_name]
    best_layer = model_data[model_data['in_distribution']==True].groupby('layer')['accuracy'].mean().idxmax()
    matrix_data = model_data[model_data['layer'] == best_layer].groupby(
        ['train_dataset', 'eval_dataset']
    )['accuracy'].mean().reset_index()

    pivot = matrix_data.pivot(index='train_dataset', columns='eval_dataset', values='accuracy')
    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.imshow(pivot.values, cmap='YlOrRd', vmin=0.4, vmax=1.0)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_yticks(range(len(pivot.index)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(pivot.index, fontsize=9)
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.0%}', ha='center', va='center', fontsize=10)
    ax.set_title(f'{model_name} - Generalization (Layer {best_layer})', fontweight='bold')
    fig.colorbar(im)
    fig.tight_layout()
    fig.savefig(f"{RESULTS_DIR}/matrix_{model_name}.png", dpi=150)
    plt.show()

In [ ]:
# Graphe 4: Comparaison avec les resultats locaux (si disponibles)
# Tu peux upload tes resultats locaux ici pour comparer
print("Pour comparer avec tes resultats locaux (Pythia):")
print("1. Upload Multi_Model_Experiments/comparison/all_results.csv dans Colab")
print("2. Ou copie-le dans ton Google Drive")
print()

local_results_path = f"{RESULTS_DIR}/local_results.csv"
if os.path.exists(local_results_path):
    df_local = pd.read_csv(local_results_path)
    df_combined = pd.concat([df, df_local], ignore_index=True)
    print(f"Combined: {len(df_combined)} results, {df_combined['model'].nunique()} models")
else:
    df_combined = df
    print("Pas de resultats locaux trouves. Les graphes ne montrent que les modeles Colab.")

# Tableau final
print("\n=== RESULTATS FINAUX ===")
for model in df_combined['model'].unique():
    m = df_combined[df_combined['model'] == model]
    m_in = m[m['in_distribution'] == True]
    print(f"{model:25s}  In-Dist: {m_in['accuracy'].mean():.1%}  Best: {m_in.groupby('algorithm')['accuracy'].mean().idxmax()}")

## 5. Sauvegarder tous les resultats

Les resultats sont automatiquement sauvegardes dans ton Google Drive:
- `LLM_Lie_Detection_Results/results_all.csv`
- `LLM_Lie_Detection_Results/summary.csv`
- `LLM_Lie_Detection_Results/*.png` (graphiques)

In [ ]:
print("Fichiers sauvegardes dans Google Drive:")
for f in sorted(Path(RESULTS_DIR).glob('*')):
    size = f.stat().st_size / 1024
    print(f"  {f.name:40s} {size:8.1f} KB")
print(f"\nChemin: {RESULTS_DIR}")